In [6]:
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np

In [7]:
# Names from json file:
names_json = ["SM","cQu8","ctu8","cQu1","ctd8","cQlMi","cpQ3","ctlTi","ctG","ctZ","ctW","ctli","ctb8","cbW","cQl3i","cQq13",
              "cptb","ctp","ctei","cpQM","ctlSi", "cQq83","cQq81","ctq1","ctu1","cQei","cQb8","cpt","ctq8","cQd1","cQq11",
              "cQd8","ctd1"]

# All 561 combinations of coefficients (i.e SM,SM; SM,cQu8; cQu8,cQu8; SM,ctu8; cQu8,ctu8; ...)
name_combinations = []
for i in range(len(names_json)):
    for a in range(i+1):
        name_combinations.append([names_json[i],names_json[a]])

In [8]:
# Open file
file = uproot.open("sample_root_files/ttH_output_919.root")
# Open events tree
events_tree = file['Events']
# Open EFTFitCoefficients branch:
eft_coeff = events_tree['EFTfitCoefficients'].array()

In [9]:
def coefficient_histogram(branch, cutoff,percent_min,percent_max):
    def weights(data):
        return [1 / len(data)] * len(data)
        
   # Different branches require different ways to make a list. Jet pt, we add all terms. Leading leptons pt, we have to do more work...
    if branch == 'Jet_pt':
        jet_pt = events_tree[branch].array()
        sum_of_jet_pt_in_each_event = []
        
        for i in range(len(jet_pt)):
            sum_of_jet_pt_in_each_event.append(sum(jet_pt[i]))
        data1 = np.array(sum_of_jet_pt_in_each_event)

    if (branch == 'Two_leading_leptons_pt') or (branch == "Two_leading_leptons_eta"):
        elec_pt = events_tree['Electron_pt'].array()
        mu_pt = events_tree['Muon_pt'].array()
        tau_pt = events_tree['Tau_pt'].array()

        if branch == 'Two_leading_leptons_eta':
            elec_eta = events_tree['Electron_eta'].array()
            mu_eta = events_tree['Muon_eta'].array()
            tau_eta = events_tree['Tau_eta'].array()
 
        leading_leptons_pt = []
        leading_leptons_eta = []

        for i in range(len(elec_pt)):
            lepton_pt = np.concatenate((elec_pt[i],mu_pt[i],tau_pt[i]), axis = 0)         # concatenating all lepton pt and lepton eta      
            order = np.argsort(lepton_pt)                                                 # order for values in ascending order
            lepton_pt_sorted = lepton_pt[order]                                           # sorting the lepton pt and lepton eta using the order given above                                          
            leading_leptons_pt.append([lepton_pt_sorted[-1],lepton_pt_sorted[-2]])        # appending largest values for lepton pt and lepton eta
            if branch == 'Two_leading_leptons_eta':
                lepton_eta = np.concatenate((elec_eta[i],mu_eta[i],tau_eta[i]), axis = 0)
                lepton_eta_sorted = lepton_eta[order]
                leading_leptons_eta.append([lepton_eta_sorted[-1],lepton_eta_sorted[-2]])
        
        if branch == 'Two_leading_leptons_pt':
            leading_leptons_pt = np.array(leading_leptons_pt)
            data1 = leading_leptons_pt[:,0]
        else:
            leading_leptons_eta = np.array(leading_leptons_eta)
            data1 = leading_leptons_eta[:,0]
    
    # Retrieving eft coefficients!
    data_above_cutoff = data1 > cutoff
    eft_coeff_above_cutoff = []
    
    for i,n in enumerate(data_above_cutoff):                                       
        if n == True:
            eft_coeff_above_cutoff.append(np.argmax(np.abs(eft_coeff[i,1:])) + 1)
            # +1 since the new list gets shifted by ignoring the first coefficient

    # Printing the interesting coefficients (any above 1%)
    percent_and_index = []
    
    for i in range(561):
        coeff_percentage = eft_coeff_above_cutoff.count(i) / len(eft_coeff_above_cutoff)
        if percent_min == percent_max: # Just to avoid errors if we are trying to find a specific percentage
            if coeff_percentage == percent_min:
               percent_and_index.append([round(coeff_percentage*100,7), i]) 
        elif (coeff_percentage > percent_min) and (coeff_percentage <= percent_max): # finds values such that percent_min < value <= percent_max
            percent_and_index.append([round(coeff_percentage*100,7), i])
    
           
    percent_and_index = np.array(percent_and_index)
    indices = np.argsort(percent_and_index[:,0])                    # This tells me the order of the rows
    percent_and_index_ordered = percent_and_index[indices][::-1] # This sorts the rows based on the indices and [::-1] makes it go in descending order
    
    for i in range(len(percent_and_index_ordered)):
        a = print(f'The pair {name_combinations[int(percent_and_index_ordered[i,1])]} with index {int(percent_and_index_ordered[i,1])} has {percent_and_index_ordered[i,0]}%')
  
    return a

In [10]:
# Values between 0 < percent <= 10%
coefficient_histogram('Jet_pt', 500, 0.05, 1)

The pair ['ctG', 'SM'] with index 36 has 33.20212%
The pair ['cQq11', 'cQq13'] with index 480 has 18.3930225%
The pair ['ctG', 'ctG'] with index 44 has 17.1349182%
The pair ['ctq1', 'ctq1'] with index 299 has 9.0588659%


In [26]:
# Values with percent = 0%
coefficient_histogram('Jet_pt', 500, 0, 0)

The pair ['ctd1', 'cQd8'] with index 559 has 0.0%
The pair ['ctd1', 'cQq11'] with index 558 has 0.0%
The pair ['ctd1', 'ctq8'] with index 556 has 0.0%
The pair ['ctd1', 'cpt'] with index 555 has 0.0%
The pair ['ctd1', 'cQb8'] with index 554 has 0.0%
The pair ['ctd1', 'cQei'] with index 553 has 0.0%
The pair ['ctd1', 'ctu1'] with index 552 has 0.0%
The pair ['ctd1', 'ctq1'] with index 551 has 0.0%
The pair ['ctd1', 'cQq81'] with index 550 has 0.0%
The pair ['ctd1', 'cQq83'] with index 549 has 0.0%
The pair ['ctd1', 'ctlSi'] with index 548 has 0.0%
The pair ['ctd1', 'cpQM'] with index 547 has 0.0%
The pair ['ctd1', 'ctei'] with index 546 has 0.0%
The pair ['ctd1', 'ctp'] with index 545 has 0.0%
The pair ['ctd1', 'cptb'] with index 544 has 0.0%
The pair ['ctd1', 'cQq13'] with index 543 has 0.0%
The pair ['ctd1', 'cQl3i'] with index 542 has 0.0%
The pair ['ctd1', 'cbW'] with index 541 has 0.0%
The pair ['ctd1', 'ctb8'] with index 540 has 0.0%
The pair ['ctd1', 'ctli'] with index 539 has 0.

In [4]:
mass = events_tree['Muon_mass'].array()

In [6]:
ak.flatten(mass)

<Array [0.106, 0.106, 0.106, ..., 0.106, 0.106, 0.106] type='143946 * float32'>